[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/relativity/gravitational_waves/hopf_projection.ipynb)

# Gravitational Waves Below the Hopf Circles

This example projects the linear gravitational waves of a closed universe, the round three-sphere, onto the two-sphere below its Hopf circles and follows a quadrupole wave travelling around the base. Great circles of the Hopf fibration fill the three-sphere at constant separation from one another, with each circle standing over a single point of the base two-sphere. Any gravitational wave that stays uniform along those circles projects onto the base without loss: its component along each circle measures how that fibre stretches, while its components across the circles deform the base itself, driving spring-bound triangle gauges and bending light rays that cross the base. All wave evolution, particle dynamics, and ray propagation run in the three-sphere on lifted fibres and project down solely for display.

Points on the unit three-sphere are modelled as antivectors, and the Hopf circles arise as the orbits of a turn acting on the same side as the smooth left-isoclinic frame motors `frame(place)`. Pulling a circle's tangent velocity back into the frame at any point along it as `frame(place).inverse() >> place.commutator(circling)` yields a single invariant step of the origin all along the circle, identifying the base two-sphere with the unit sphere of steps at the origin. Turning a base tangent direction by a quarter turn in the plane orthogonal to the base point via `turning(base).commutator(direction)` gives the horizontal step across the circles whose angle the Hopf projection doubles, while pairing the strain against the round ***duality map*** `(strain(step) & step) / (2 * (polar(step) & step))` reads fractional length stretch along any fibre or horizontal direction. First-person ray tracing of the unperturbed three-sphere appears in [rendering.ipynb](rendering.ipynb), and the interior crystal view of the standing squash mode appears in [waves.ipynb](waves.ipynb).

In quaternion and Riemannian submersion notation, the Hopf projection maps a unit quaternion $q \in S^3$ to the unit imaginary quaternion $\pi(q) = \bar q\,\mathbf{i}\,q \in S^2$, which stays constant along every left coset $e^{\mathbf{i}t} q$ and doubles horizontal angles so orthogonal fibres at separation $\pi/2$ project to antipodes; a fibre-invariant perturbation $h_{ab}$ on $S^3$ decomposes under Kaluza–Klein reduction along the vertical Killing field $\xi$ into a scalar dilaton $h(\xi, \xi)$ measuring fibre stretch and a horizontal tensor on $S^2$ of curvature $4$.

In [ ]:
# In Colab, numga and its examples, from GitHub.
import sys

if "google.colab" in sys.modules:
    %pip install -q "numga[examples] @ https://github.com/EelcoHoogendoorn/numga/archive/HEAD.tar.gz"

In [ ]:
%matplotlib inline
from collections.abc import Iterator

import numpy as np
from numga import Algebra, NumpyContext, stack
from numga.sparse import spdiag
from numga.topology import Topology

from examples.util.animation import show_states
from examples.relativity.gravitational_waves import numerical, render, scenarios

ga = Algebra("x+y+z+w+")
mv = NumpyContext(ga).multivector
Scalar = ga.gatype.scalar()                                                     # Scalar+
Amount = ga.gatype.antiscalar()                                                 # Antiscalar-
Plane = ga.gatype.vector()                                                      # Vector+
Through = Plane.derive.within(Algebra("x+y+z+"))                                # Vector+
Point = ga.gatype.antivector()                                                  # Antivector-
Step = Through.derive.dual                                                      # Antivector-
Motor = ga.gatype.rotor()                                                       # Rotor+
Rate = ga.gatype.bivector()                                                     # Bivector+
Strain = ga.gatype((Through, Step))                                             # Plane+ <- Point-
Turning = ga.gatype((Rate, Step))                                               # Rate+ <- Point-
Rolling = ga.gatype((Rate, Step))                                               # Rate+ <- Point-
Stretching = ga.gatype((Step, Step))                                            # Point- <- Point-
Translating = ga.gatype((Step, Through))                                        # Point- <- Plane+

# Equatorial plane `w` and the origin, its pole on the three-sphere.
equator = mv.w                                                                  # [] Plane+
origin = equator.dual()                                                         # [] Point-
# Step of the origin across a plane through it; its inverse sends each step to its polar plane as a duality map.
translating = origin.commutator(equator ^ Through).cast(Translating)            # [] Point- <- Plane+
polar = translating.inverse()                                                   # [] Plane+ <- Point-
moving = equator ^ polar(Step)                                                  # [] Rate+ <- Point-
settling = 1e-10

## The frame at each place and the Hopf circles

In four dimensions, multiplying a bivector rate by the extrinsic pseudoscalar produces the rate in the orthogonal plane, splitting every rate into an invariant half and a flipped half. Each half turns by equal angles in two orthogonal planes, moving every point of the sphere the same distance along a great circle. Keeping the pseudoscalar-invariant half of `place / origin` with unit scalar part in the other half gives the smooth frame motor `frame(place)` everywhere on the three-sphere. One-parameter turns generated by a rate from that same half sweep out disjoint great circles covering the sphere, its Hopf circles, while `turning` rotates tangent steps by a quarter turn in the orthogonal plane across the fibres.

In [ ]:
pseudoscalar = mv.xyzw                                                          # [] Pseudoscalar+
turning = moving * pseudoscalar                                                 # [] Rate+ <- Point-


def frame(places: Point) -> Motor:
    """Smooth left-isoclinic frame motors `Motor+` carry the origin to `places` by turning equally in the great-circle
    plane and its orthogonal complement."""
    return ((1 + pseudoscalar) * (places.normalized() / origin) + (1 - pseudoscalar)) / 2

## The base below the circles

Twelve guardrail decagons of the 600-cell follow Hopf circles generated by a one-tenth turn that maps the 600-cell onto itself, and their base points form the twelve vertices of an icosahedron on the base two-sphere. Because the Hopf map doubles horizontal angles, lifting any base point moves the origin horizontally across the circles by half the angle from the origin's own base point.

In [ ]:
cell_topology, cell_vertices = scenarios.six_hundred_cell()
V, E, F, T = cell_topology.n_elements
turn = scenarios.circle_turn(cell_vertices, scenarios.n_guardrail_sides)        # [] Motor+
circling = turn.log()                                                           # [] Rate+


def base_points(places: Point) -> Point:
    """Pulling the fibre velocity under `circling` into the frame at `places` gives the invariant Hopf base point `Point-`."""
    circle_directions = places.commutator(circling)                             # Point-
    return (frame(places).inverse() >> circle_directions).cast(Step).normalized()


def crossing(bases: Point, directions: Point) -> Point:
    """Quarter-turns of base tangent `directions` in the plane orthogonal to `bases` give horizontal steps `Point-` across
    the Hopf circles."""
    return turning(bases).commutator(directions)

In [ ]:
origin_base = base_points(origin)                                               # [] Point-


def lifts(bases: Point) -> Point:
    """Moving the origin horizontally across the Hopf circles by half the base angle lifts `bases` to points `Point-` on
    their fibres."""
    cosines = polar(origin_base) & bases                                        # Scalar+
    directions = (bases - origin_base * cosines).normalized()                   # Point-
    lifting = (moving(crossing(origin_base, directions)) * (-cosines.arccos() / 4)).exp()   # Motor+
    return lifting >> origin


decagons = scenarios.guardrails(cell_vertices, scenarios.n_guardrail_sides)
n_decagons = len(decagons)
decagon_bases = base_points(cell_vertices.batch()[decagons[:, 0]])              # [n_decagons] Point-

## The travelling quadrupole

Spherical Regge calculus on the 600-cell supplies the edge-length stiffness, relativity's kinetic mass map, and the mass-orthogonal projection onto transverse-traceless waves; the term ***simplex gradient*** here means the plane through each corner's opposite facet scaled to pair to one at that corner. Superposing the standing squash with a second squash rotated by an eighth of a turn in the equatorial plane and lagging by a quarter period produces a quadrupole wave that travels steadily around the base while remaining uniform along every Hopf circle. Stepping the history and evaluating the vertex strain tables across all stored levels takes about seven seconds.

In [ ]:
swing = 0.085
lags = np.array([0, np.pi / 2])                                                 # [2]
n_steps_per_swing = 40
n_settling_swings = 8
light_step = 0.02
n_light_steps = int(np.pi / 2 / light_step)

In [ ]:
chords = cell_topology.d[0] * cell_vertices                                     # Point-[E]
rest = chords.scalar_norm_squared()                                             # Scalar+[E]
corners = cell_topology.incidence[3, 0] * cell_vertices                         # [4] Point-[T]
facing = cell_topology.off[3, 0] * cell_vertices                                # [3, 4] Point-[T]
facets = facing[0] & facing[1] & facing[2]                                      # [4] Plane-[T]
gradients = facets / (facets & corners)                                         # [4] Plane+[T]
radial = gradients.sum(axis=0)                                                  # Plane+[T]
centred_gradients = gradients - radial / 4                                      # [4] Plane+[T]
tails, heads = cell_topology.corners[3, 1]
off_tails, off_heads = cell_topology.complements[3, 1]

flat_basis = -(centred_gradients[tails] * (centred_gradients[heads] & Point)
               + centred_gradients[heads] * (centred_gradients[tails] & Point)) / 2   # [6] (Plane+ <- Point-)[T]
flat_map = (spdiag(flat_basis) * cell_topology.incidence[3, 1]).sum(axis=0)     # (Plane+ <- Point-)[T <- E]
flat = radial * (radial & Point) + flat_map * rest                              # (Plane+ <- Point-)[T]
spans = corners[1:] - corners[0]                                                # [3] Point-[T]
joins = corners[0] & corners[1] & corners[2] & corners[3]                       # Amount-[T]
volumes = ((flat(spans[0]) ^ flat(spans[1]) ^ flat(spans[2]) ^ radial) & joins).abs().square_root() / 6   # Scalar+[T]
pairings = centred_gradients[:, None] & flat.batch().inverse().field()(centred_gradients[None, :])   # [4, 4] Scalar+[T]
tet_masses = volumes / 4 * (pairings[tails[:, None], tails[None, :]] * pairings[heads[:, None], heads[None, :]]
                            + pairings[tails[:, None], heads[None, :]] * pairings[heads[:, None], tails[None, :]])   # [6, 6] Scalar+[T]
edges = cell_topology.incidence[3, 1]                                           # [6] Scalar+[T <- E]
masses = (edges[:, None].adjugate() * spdiag(tet_masses * Scalar.dual()) * edges[None, :]).sum(axis=0).sum(axis=0)   # (Amount- <- Scalar+)[E <- E]

cone_basis = -(gradients[tails] * (gradients[heads] & Point) + gradients[heads] * (gradients[tails] & Point)) / 2   # [6] (Plane+ <- Point-)[T]
cone_map = (spdiag(cone_basis) * cell_topology.incidence[3, 1]).sum(axis=0)     # (Plane+ <- Point-)[T <- E]
cone = radial * (radial & Point) + cone_map * rest                              # (Plane+ <- Point-)[T]
cone_pairings = gradients[:, None] & cone.batch().inverse().field()(gradients[None, :])   # [4, 4] Scalar+[T]
first_off, second_off = off_tails[:, None], off_heads[:, None]
tail_ends, head_ends = tails[None, :], heads[None, :]
cross_pairings = (cone_pairings[first_off, tail_ends] * cone_pairings[second_off, head_ends]
                  + cone_pairings[first_off, head_ends] * cone_pairings[second_off, tail_ends]) / 2   # [6, 6] Scalar+[T]
self_pairings = (cone_pairings[first_off, tail_ends] * cone_pairings[first_off, head_ends] / cone_pairings[first_off, first_off]
                 + cone_pairings[second_off, tail_ends] * cone_pairings[second_off, head_ends] / cone_pairings[second_off, second_off])
edge_pairings = cone_pairings[first_off, second_off]                            # [6, 1] Scalar+[T]
sines = (cone_pairings[first_off, first_off] * cone_pairings[second_off, second_off] - edge_pairings.squared()).square_root()   # [6, 1] Scalar+[T]
angle_changes = (cross_pairings - edge_pairings * self_pairings / 2) / sines    # [6, 6] Scalar+[T]
roots = edges * (rest * (1 - rest / 4)).square_root()                           # [6] Scalar+[T]
stiffness = (edges[:, None].adjugate() * spdiag(angle_changes / roots[:, None] * Scalar.dual()) * edges[None, :]).sum(axis=0).sum(axis=0)   # (Amount- <- Scalar+)[E <- E]

In [ ]:
moves = spdiag(2 * chords.dot(Point)) * cell_topology.d[0]                      # (Scalar+ <- Point-)[E <- V]
reach = moves.adjugate()(masses(moves))                                         # (Plane+ <- Point-)[V <- V]
reach = reach + settling * numerical.largest(reach) * Point.dual()


def wave_projection(changes: Scalar) -> Scalar:
    """Mass-orthogonal projection subtracts from `changes` `Scalar+[E]` the nearest vertex-motion perturbation."""
    return changes - moves(reach.solve(moves.adjugate()(masses(changes))))


def round_steps(graph: Topology, places: Point) -> Point:
    """Pulling each edge's head back into its tail's frame gives the edge's tangent step `Point-[E]` at the origin."""
    edge_tails, edge_heads = graph.incidence[1, 0] * places                     # Point-[E], Point-[E]
    return (frame(edge_tails).inverse() >> edge_heads).cast(Step)


eighth = (mv.xy * (-np.pi / 8)).exp()                                           # [] Motor+
stretched = stack((mv.x, eighth >> mv.x))                                       # [2] Plane+
squeezed = stack((mv.y, eighth >> mv.y))                                        # [2] Plane+
steps = round_steps(cell_topology, cell_vertices)                               # Point-[E]
shares = ((stretched & steps).squared() - (squeezed & steps).squared()) / (polar(steps) & steps)   # [2] Scalar+[E]
patterns = wave_projection(2 * rest * shares)                                   # [2] Scalar+[E]
squash = patterns[0]                                                            # Scalar+[E]
squash_squared = (squash & stiffness(squash)).sites.sum() / (squash & masses(squash)).sites.sum()   # [] Scalar+
print(f"the squash's frequency squared: {float(squash_squared.to_array()):.3f}")

In [ ]:
def waves_stepped(strains: Scalar, rates: Scalar, step: float) -> Iterator[tuple[Scalar, Scalar]]:
    """Yields squared edge-length perturbations and synchronized rates `(Scalar+[E], Scalar+[E])` across projected
    leapfrog steps."""
    restoring = wave_projection(masses.solve(stiffness(strains)))               # Scalar+[E]
    prior_rates = rates + (step / 2) * restoring                                # Scalar+[E]
    while True:
        next_rates = prior_rates - step * restoring                             # Scalar+[E]
        yield strains, (prior_rates + next_rates) / 2
        strains, prior_rates = strains + step * next_rates, next_rates
        restoring = wave_projection(masses.solve(stiffness(strains)))


frequency = np.sqrt(numerical.scalar_value(squash_squared))
period = 2 * np.pi / frequency
level_step = period / n_steps_per_swing
first_time = -(n_settling_swings + 1) * period
n_levels = int(np.ceil((period + n_light_steps * light_step - first_time) / level_step)) + 2
phases = frequency * first_time - lags                                          # [2]
strains = swing * (np.cos(phases) * patterns).sum(axis=0)                       # Scalar+[E]
rates = -swing * frequency * (np.sin(phases) * patterns).sum(axis=0)            # Scalar+[E]
levels = [state for _, state in zip(range(n_levels), waves_stepped(strains, rates, level_step))]

tet_frames = frame(cell_topology.incidence[3, 0].mean(axis=0) * cell_vertices)   # Motor+[T]
counts = (cell_topology.incidence[3, 0].adjugate() * mv.scalar(np.ones((T, 1))).field()).sum(axis=0)   # Scalar+[V]


def strain_field(changes: Scalar) -> tuple[Strain, Turning]:
    """Pulling each tetrahedron's cone-metric perturbation into its midpoint frame and averaging to vertices gives the
    vertex strain `(Plane+ <- Point-)[V]` and the tetrahedral curl turn `(Rate+ <- Point-)[T]`."""
    frame_steps = tet_frames >> Step                                            # (Point- <- Point-)[T]
    tet_strains = (tet_frames.inverse() >> (cone_map * changes)(frame_steps)).cast(Strain)   # (Plane+ <- Point-)[T]
    at_vertices = (cell_topology.incidence[3, 0].adjugate() * tet_strains).sum(axis=0) / counts   # (Plane+ <- Point-)[V]
    corner_planes = (tet_frames.inverse() >> gradients).cast(Through)           # [4] Plane+[T]
    corner_strains = cell_topology.incidence[3, 0] * at_vertices                # [4] (Plane+ <- Point-)[T]
    return at_vertices, (corner_planes ^ corner_strains).sum(axis=0) / 2


level_fields = [strain_field(changes) for changes, _ in levels]
level_strains = stack([strain.batch() for strain, _ in level_fields])           # [n_levels, V] Plane+ <- Point-
level_turns = stack([turn_map.batch() for _, turn_map in level_fields])         # [n_levels, T] Rate+ <- Point-
level_strain_rates = stack([strain_field(rate)[0].batch() for _, rate in levels])   # [n_levels, V] Plane+ <- Point-
level_traces = translating(level_strains).trace()                               # [n_levels, V] Scalar+

In [ ]:
def wave_fields(tetrahedra: np.ndarray, places: Point, time: float) -> tuple[Strain, Strain, Scalar, Turning]:
    """Simplex-gradient corner weights and linear time weights interpolate the strain, strain rate, trace, and
    tetrahedral turn at `places` `[...] Point-[V_sites]` inside `tetrahedra`."""
    level = (time - first_time) / level_step
    early_index, share = int(level), level - int(level)
    tet_reading = cell_topology.reading(tetrahedra, 3)[..., None]               # [..., 1] Scalar+[V_sites <- T]
    place_gradients = tet_reading * gradients                                   # [..., 4] Plane+[V_sites]
    weights = (place_gradients & places[..., None]) / (place_gradients.sum(axis=-1) & places)[..., None]   # [..., 4] Scalar+[V_sites]

    def blended_field(values):
        """Linear time and corner interpolation evaluates vertex tables `[n_levels, V]` at `places`."""
        time_slice = (values[early_index] * (1 - share) + values[early_index + 1] * share).field()   # [V]
        return (weights * (tet_reading * (cell_topology.incidence[3, 0] * time_slice))).sum(axis=-1)

    turns = (level_turns[early_index] * (1 - share) + level_turns[early_index + 1] * share).field()   # (Rate+ <- Point-)[T]
    return (blended_field(level_strains), blended_field(level_strain_rates),
            blended_field(level_traces), tet_reading[..., 0] * turns)

In [ ]:
def wave_connection(tetrahedra: np.ndarray, places: Point, time: float) -> tuple[Rolling, Stretching, Stretching]:
    """Wave fields at `places` `[...] Point-[V_sites]` assemble the perturbed frame-rolling connection
    `[...] (Rate+ <- Point-)[V_sites]`, spatial stretch, and slice-rate squeeze `[...] (Point- <- Point-)[V_sites]`."""
    strain, strain_rate, trace, turn_map = wave_fields(tetrahedra, places, time)
    stretch = translating(strain(Step))                                         # [...] (Point- <- Point-)[V_sites]
    connection_rate = moving(Step + stretch / 2) + turning(Step - 2 * stretch + Step * (trace / 2)) + turn_map(Step)
    return connection_rate, stretch, translating(strain_rate) / 2

## The stretch along the circles

Pairing the strain along each circle's fibre direction against the round duality map measures how much that circle stretches. Averaging across seven evenly spaced points along each fibre samples between the tenfold symmetry steps of the 600-cell so the triangulation's discrete facets do not imprint on the base icosphere.

In [ ]:
n_circle_samples = 7

In [ ]:
sphere_topology, bases = scenarios.base_sphere()
pick = numerical.picking(cell_topology, cell_vertices)


def stretches(bases: Point, directions: Point, time: float) -> Scalar:
    """Averaging the relative metric stretch along `directions` around `n_circle_samples` fibre points gives the
    circle-averaged stretch `Scalar+[V_bases]`."""
    shares = np.arange(n_circle_samples) * scenarios.n_guardrail_sides / n_circle_samples   # [n_circle_samples]
    turns = (circling * shares[:, None]).exp()                                  # [n_circle_samples] Motor+
    places = turns >> lifts(bases)[None]                                        # [n_circle_samples] Point-[V_bases]
    strain, _, _, _ = wave_fields(pick(places), places, time)                   # [n_circle_samples] (Plane+ <- Point-)[V_bases]
    readings = (strain(directions) & directions) / (2 * (polar(directions) & directions))   # [n_circle_samples] Scalar+[V_bases]
    return readings.sum(axis=0) / n_circle_samples

## Triangle gauges

Inside each face of the icosahedron formed by the decagon base points sits a triangle of unit-mass atoms drawn toward the face centre and lifted horizontally from the central fibre so its bonds run nearly across the circles. Springs and dashpots along the bonds set each triangle's shear mode above the wave frequency and damp it critically. Near the poles the wave's pattern turns, and the angular momentum it carries spins each damped gauge up until the gauge turns with the pattern; with the shear mode detuned from the wave, the deformation that turns along with the gauge stays off resonance, and its bonds hold a modest stretch.

In [ ]:
gauge_share = 0.4
gauge_tuning = 1.4
gauge_damping = 1.0
n_frames = 75
atom_step = 0.01

In [ ]:
icosahedron = numerical.base_hull(decagon_bases)
V_ico, E_ico, F_ico = icosahedron.n_elements
face_corners = icosahedron.incidence[2, 0] * decagon_bases.field()              # [3] Point-[F_ico]
middles = face_corners.sum(axis=0).normalized()                                 # Point-[F_ico]
atom_bases = (middles + gauge_share * (face_corners - middles)).normalized()    # [3] Point-[F_ico]
# Lifting horizontally from each face midpoint by half the base angle places the triangle bonds across the fibres.
cosines = polar(middles) & atom_bases                                           # [3] Scalar+[F_ico]
directions = (atom_bases - middles * cosines).normalized()                      # [3] Point-[F_ico]
lift_moves = (moving(crossing(middles, directions)) * (-cosines.arccos() / 4)).exp()   # [3] Motor+[F_ico]
face_atoms = (frame(lifts(middles)) * lift_moves) >> origin                     # [3] Point-[F_ico]
atoms = stack(tuple(face_atoms.batch()), axis=-1).reshape(-1).field()           # Point-[V_atoms]
bonds = Topology.loops([3] * F_ico)

In [ ]:
def massive(bonds: Topology, rest: Scalar, spring_stiffness: Scalar, spring_damping: Scalar, starts: Point,
            velocities: Point, start: float, step: float, times: np.ndarray) -> Iterator[tuple[Point, Scalar]]:
    """Yields particle positions `Point-[V_atoms]` and bond proper strains `Scalar+[E_bonds]` at `times` as spring-bound
    particles step through the wave connection by symplectic Euler."""
    V_atoms, E_bonds = bonds.n_elements
    positions, velocities, lengths, time = starts.batch(), velocities.batch(), rest, start
    bond_tails = bonds.incidence[1, 0][0]                                       # Scalar+[E_bonds <- V_atoms]
    for target in times:
        while time < target - step / 2:
            places = positions.field()                                          # Point-[V_atoms]
            connection_rate, stretch, squeeze = wave_connection(pick(positions), places, time)   # (Rate+ <- Point-)[V_atoms], (Point- <- Point-)[V_atoms], (Point- <- Point-)[V_atoms]
            proper = origin.commutator((bond_tails * connection_rate)(round_steps(bonds, places))).cast(Step)   # Point-[E_bonds]
            previous_lengths, lengths = lengths, proper.norm()                  # Scalar+[E_bonds], Scalar+[E_bonds]
            pulls = (spring_stiffness * (lengths - rest) + spring_damping * (lengths - previous_lengths) / step) / lengths   # Scalar+[E_bonds]
            forces = -(bonds.d[0].adjugate() * (pulls * proper)).batch()        # [V_atoms] Point-
            connection_rate, stretch, squeeze = connection_rate.batch(), stretch.batch(), squeeze.batch()   # [V_atoms] Rate+ <- Point-, Point- <- Point-, Point- <- Point-
            motion = velocities - stretch(velocities) / 2                       # [V_atoms] Point-
            carried_velocities = velocities - velocities.commutator(connection_rate(motion)).cast(Step) * step
            velocities = carried_velocities - squeeze(carried_velocities) * step + forces * step   # [V_atoms] Point-
            rolling = velocities - stretch(velocities) / 2                      # [V_atoms] Point-
            rolls = frame(positions) * (moving(rolling) * (-step / 2)).exp()    # [V_atoms] Motor+
            positions = rolls >> origin                                         # [V_atoms] Point-
            time += step
        yield positions.field(), lengths / rest - 1


V_atoms, E_bonds = bonds.n_elements
# Unit masses on an equilateral triangle shear at a squared frequency of three halves the bond stiffness.
shear_frequency = gauge_tuning * frequency
bond_stiffness = mv.scalar(np.full((E_bonds, 1), 2 / 3 * shear_frequency ** 2)).field()   # Scalar+[E_bonds]
bond_damping = mv.scalar(np.full((E_bonds, 1), 2 / 3 * (2 * gauge_damping * shear_frequency))).field()   # Scalar+[E_bonds]
bond_steps = round_steps(bonds, atoms)                                          # Point-[E_bonds]
bond_rest = bond_steps.norm()                                                   # Scalar+[E_bonds]
times = np.arange(n_frames) * period / n_frames                                 # [n_frames]
at_rest = mv(Through, np.zeros((V_atoms, 3))).dual().field()                    # Point-[V_atoms]
states = massive(bonds, bond_rest, bond_stiffness, bond_damping, atoms, at_rest,
                 times[0] - n_settling_swings * period, atom_step, times)

## A bundle of light

At every frame, a narrow fan of rays launches horizontally across the Hopf circles from the north-pole fibre toward the viewer-facing pole. Without gravitational waves, every horizontal geodesic crosses the base along a meridian to focus at the south pole after a quarter turn of the three-sphere; the travelling quadrupole deflects the bundle across the base.

In [ ]:
n_rays = 7
light_spread = np.radians(18)

In [ ]:
source, target = mv.z.dual(), mv.y.dual()                                       # [] Point-, [] Point-
aim = (target - source * (polar(source) & target)).normalized()                 # [] Point-
fan = (mv.xy * (-np.linspace(-light_spread / 2, light_spread / 2, n_rays) / 2)).exp()   # [n_rays] Motor+
headings = crossing(source, fan >> aim).normalized()                            # [n_rays] Point-
starts = lifts(source[None]).broadcast_to((n_rays,))                            # [n_rays] Point-


def light(starts: Point, headings: Point, start: float, step: float, n_steps: int) -> Iterator[Point]:
    """Yields ray positions `[n_rays] Point-` across `n_steps` midpoint geodesic steps of arc length `step`."""

    def turning_and_rolling(places: Point, heading: Point, time: float) -> tuple[Point, Point]:
        """Wave connection and slice stretch at `places` give the heading derivative and round rolling step `[n_rays] Point-`."""
        connection_rate, stretch, squeeze = wave_connection(pick(places), places.field(), time)   # (Rate+ <- Point-)[V_rays], (Point- <- Point-)[V_rays], (Point- <- Point-)[V_rays]
        connection_rate, stretch, squeeze = connection_rate.batch(), stretch.batch(), squeeze.batch()   # [n_rays] Rate+ <- Point-, Point- <- Point-, Point- <- Point-
        rolling = heading - stretch(heading) / 2                                # [n_rays] Point-
        return -heading.commutator(connection_rate(rolling)).cast(Step) - squeeze(heading), rolling

    positions, time = starts, start
    for _ in range(n_steps):
        start_change, start_rolling = turning_and_rolling(positions, headings, time)
        half_headings = (headings + start_change * (step / 2)).normalized()     # [n_rays] Point-
        half_rolls = frame(positions) * (moving(start_rolling) * (-step / 4)).exp()   # [n_rays] Motor+
        half_positions = half_rolls >> origin                                   # [n_rays] Point-
        change, rolling = turning_and_rolling(half_positions, half_headings, time + step / 2)
        headings = (headings + change * step).normalized()                      # [n_rays] Point-
        rolls = frame(positions) * (moving(rolling) * (-step / 2)).exp()        # [n_rays] Motor+
        positions = rolls >> origin                                             # [n_rays] Point-
        time += step
        yield positions

## The base from outside

Projecting the fibre stretch, the triangle gauges, and the light fan onto the base two-sphere at each frame displays the travelling quadrupole from outside. Stepping the atoms, tracing the rays, and rendering the frames takes about half a minute.

In [ ]:
n_arc_points = 12
duration_ms = 40

In [ ]:
def base_frames() -> Iterator[tuple]:
    """Yields fibre stretches, projected atom positions, bond arcs, bond strains, and ray paths on the base sphere."""
    arc_shares = np.linspace(0, 1, n_arc_points)                                # [n_arc_points]
    for time, (positions, bond_strains) in zip(times, states):
        bond_tails, bond_heads = bonds.incidence[1, 0] * positions              # Point-[E_bonds], Point-[E_bonds]
        arc_points = (bond_tails.batch()[:, None] * (1 - arc_shares) + bond_heads.batch()[:, None] * arc_shares).normalized()   # [E_bonds, n_arc_points] Point-
        light_places = light(starts, headings, time, light_step, n_light_steps)
        paths = stack([starts, *light_places], axis=-1)                         # [n_rays, n_light_steps + 1] Point-
        yield (stretches(bases, bases, time), base_points(positions.batch()), base_points(arc_points),
               bond_strains.batch(), base_points(paths))


base_history = show_states(base_frames(), render.base_views, sphere_topology, bases, decagon_bases,
                           name="gravitational_waves_quadrupole", duration_ms=duration_ms);

Surface shading shows the stretch of the Hopf circle above each base point from shortened blue to lengthened red. Coloured dots mark the twelve decagon fibres, triangle gauge bonds appear as great-circle projections coloured by proper strain, one atom of each gauge marked green so that its turning shows, and dark curves trace the deflected light bundle.

In [ ]:
# checks
first_stretches = base_history[0][0]                                            # Scalar+[V_sphere]
lifted_bases = lifts(bases)                                                     # Point-[V_sphere]
strain, _, _, _ = wave_fields(pick(lifted_bases), lifted_bases, 0.0)            # (Plane+ <- Point-)[V_sphere]
single_point_stretches = (strain(bases) & bases) / (2 * (polar(bases) & bases))   # Scalar+[V_sphere]
assert ((single_point_stretches - first_stretches).abs() < 1e-6 * first_stretches.abs().max()).all()
bases_of_atoms = stack(tuple(atom_bases.batch()), axis=-1).reshape(-1)          # [V_atoms] Point-
assert (((polar(base_points(atoms.batch())) & bases_of_atoms) - 1).abs() < 1e-10).all()
assert ((polar(base_points(starts)) & headings).abs() < 1e-10).all()